# LoRA Adapter Merge & Submit

Merges two LoRA adapters (baseline + new) via weighted average of safetensors weights.

**Attach both adapters as Kaggle Model inputs before running.**

In [ ]:
import os, json, shutil, zipfile, glob
from safetensors.torch import load_file, save_file
import torch

WORK_DIR = "/kaggle/working"
REQUIRED = {"adapter_config.json", "adapter_model.safetensors"}

# ============================================================
# CONFIGURE: paths to both adapters
# Update these to match your Kaggle model input paths
# ============================================================
ADAPTER_A_PATH = "/kaggle/input/models/manish756/nvidia-adapter/transformers/adapter_recreating_baseline_model/22"  # baseline (0.86)
ADAPTER_B_PATH = "/kaggle/input/models/manish756/nvidia-adapter/transformers/YOUR_NEW_ADAPTER_VERSION_HERE"         # new v12 (0.86, +eq_guess)

# Merge weight: 0.0 = pure A (baseline), 1.0 = pure B (new)
# Try 0.5 first (equal blend), then 0.3 if it regresses
ALPHA = 0.5

print(f"Adapter A (baseline): {ADAPTER_A_PATH}")
print(f"Adapter B (new):      {ADAPTER_B_PATH}")
print(f"Merge alpha:          {ALPHA}  (merged = (1-α)*A + α*B)")

In [ ]:
# ============================================================
# LOAD both adapter weights
# ============================================================
def load_adapter(path):
    """Load adapter from directory or zip."""
    safetensors_path = os.path.join(path, "adapter_model.safetensors")
    config_path = os.path.join(path, "adapter_config.json")
    
    if not os.path.exists(safetensors_path):
        raise FileNotFoundError(f"adapter_model.safetensors not found in {path}")
    
    weights = load_file(safetensors_path)
    with open(config_path) as f:
        config = json.load(f)
    
    return weights, config

print("Loading adapter A (baseline)...")
weights_a, config_a = load_adapter(ADAPTER_A_PATH)
print(f"  Keys: {len(weights_a)}, Total params: {sum(v.numel() for v in weights_a.values()):,}")

print("Loading adapter B (new)...")
weights_b, config_b = load_adapter(ADAPTER_B_PATH)
print(f"  Keys: {len(weights_b)}, Total params: {sum(v.numel() for v in weights_b.values()):,}")

# Verify same architecture
assert weights_a.keys() == weights_b.keys(), "Adapter keys mismatch!"
assert config_a["r"] == config_b["r"], f"Rank mismatch: {config_a['r']} vs {config_b['r']}"
assert set(config_a["target_modules"]) == set(config_b["target_modules"]), "Target modules mismatch!"
print("\nArchitecture match confirmed ✓")

In [ ]:
# ============================================================
# MERGE: weighted average of all weight tensors
# merged = (1 - ALPHA) * A + ALPHA * B
# ============================================================
print(f"Merging with alpha={ALPHA} ...")
print(f"  Formula: merged = {1-ALPHA:.1f} * baseline + {ALPHA:.1f} * new\n")

merged_weights = {}
for key in weights_a:
    wa = weights_a[key]
    wb = weights_b[key]
    merged_weights[key] = (1.0 - ALPHA) * wa + ALPHA * wb

# Sanity: check a few layers for non-trivial difference
diffs = []
for key in list(weights_a.keys())[:10]:
    diff = (weights_a[key] - weights_b[key]).abs().mean().item()
    diffs.append((key.split('.')[-2] + '.' + key.split('.')[-1], diff))

print("Weight differences (first 10 layers):")
for name, d in diffs:
    print(f"  {name:>30s}: mean_abs_diff = {d:.6f}")

print(f"\nMerged {len(merged_weights)} tensors ✓")

In [ ]:
# ============================================================
# SAVE merged adapter
# ============================================================
merged_safetensors = os.path.join(WORK_DIR, "adapter_model.safetensors")
merged_config = os.path.join(WORK_DIR, "adapter_config.json")

# Save weights
save_file(merged_weights, merged_safetensors)
size_mb = os.path.getsize(merged_safetensors) / 1024 / 1024
print(f"Saved merged weights: {size_mb:.1f} MB")

# Use config from adapter A (they should be identical)
shutil.copy2(os.path.join(ADAPTER_A_PATH, "adapter_config.json"), merged_config)
print(f"Copied adapter_config.json")

# Free memory
del weights_a, weights_b, merged_weights
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("Memory freed ✓")

In [ ]:
# ============================================================
# VERIFY adapter_config.json
# ============================================================
with open(merged_config) as f:
    cfg = json.load(f)

print("Merged adapter config:")
print(f"  rank:           {cfg.get('r')}")
print(f"  alpha:          {cfg.get('lora_alpha')}")
print(f"  dropout:        {cfg.get('lora_dropout')}")
print(f"  target_modules: {cfg.get('target_modules')}")
print(f"  peft_type:      {cfg.get('peft_type')}")

In [ ]:
# ============================================================
# PACKAGE submission.zip
# ============================================================
SUB_ZIP = os.path.join(WORK_DIR, "submission.zip")

if os.path.exists(SUB_ZIP):
    os.remove(SUB_ZIP)

with zipfile.ZipFile(SUB_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(REQUIRED):
        fpath = os.path.join(WORK_DIR, fname)
        zf.write(fpath, arcname=fname)

with zipfile.ZipFile(SUB_ZIP) as zf:
    contents = zf.namelist()

sub_mb = os.path.getsize(SUB_ZIP) / 1024 / 1024

print(f"{'='*55}")
print(f"  SUBMISSION READY")
print(f"{'='*55}")
print(f"  File     : {SUB_ZIP}")
print(f"  Size     : {sub_mb:.1f} MB")
print(f"  Contents : {contents}")
print(f"  Merge    : {1-ALPHA:.0%} baseline + {ALPHA:.0%} new")
assert set(contents) == REQUIRED, f"Wrong files: {contents}"